# Handover — DDI link prediction

Start here. This notebook walks through the project end to end: what the data is,
what the central result is, and how to reproduce it. Run it top to bottom; it writes
nothing and takes a few minutes on CPU.

**The project in three sentences.** We predict drug-drug interactions as link
prediction on the OGB-ddi benchmark, using a 3-layer GCN over biomedical text
embeddings. We then ask whether the benchmark score reflects pharmacology or an
artefact of graph structure. It is largely an artefact, and most of this notebook
is the evidence for that.

**The files**

| File | What it is |
|---|---|
| `DATA.md` | Reference for every array on disk. Read it next. |
| `ddi_toolkit.py` | Analysis functions. Loads data, rescores pairs, runs diagnostics. |
| `01_freeze_splits.py` | How the splits were built. **Do not run** — it would invalidate stored results. |
| `05_gcn_experiment.py` | Training + evaluation on the three negative sets. Needs a GPU. |

Supervisors: Paraskevi Nousi (Vivi), Maxim Samarin — SDSC.

---
## 1. Setup and sanity checks

If anything below fails, stop and fix it before trusting any later cell.

In [8]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "/home/renku/work/DDI_ML")

import numpy as np
from ddi_toolkit import *

print("device:", DEVICE)   # 'cpu' on Renku is expected — training happens on Colab

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
device: cpu


In [2]:
check_files()

g = load_graph()
print("\ngraph:", g.summary())

DATA   /home/renku/work/data  OK
FROZEN /home/renku/work/data/frozen_r5  OK

split files:
  cres_dd_test.npy           (172, 2)      
  cres_dd_train.npy          (1853, 2)     
  cres_dd_val.npy            (39, 2)       
  cres_test.npy              (811, 2)      
  cres_train.npy             (2834, 2)     
  cres_val.npy               (404, 2)      
  degmatched_neg_test.npy    (810, 2)      
  ogb_test_neg.npy           (95599, 2)    
  ogb_test_pos.npy           (133117, 2)   
  ogb_train_pos.npy          (1066276, 2)  
  ogb_val_neg.npy            (101882, 2)   
  ogb_val_pos.npy            (133263, 2)   
  random_neg_test.npy        (133117, 2)   
  random_neg_train.npy       (1066276, 2)  
  random_neg_val.npy         (20000, 2)    

feature files:
  bothwhite    bothwhite_embeddings.npy         (4267, 1536)
  cbwhite      cbwhite_embeddings.npy           (4267, 768)
  chemberta    chemberta_embeddings.npy         (4267, 768)
  gpt          gpt_embeddings.npy               (3481

Note the median degree: **446** out of 4,266 possible neighbours. The graph is 14.7%
dense. That density is the root of everything that follows — in a graph this
connected, "is this a popular drug?" is already a decent predictor of "does this pair
interact?".

Next: confirm the CRESCENDDI conflict removal actually happened. Every overlap count
must be zero. If it is not, you are on the old `frozen/` directory and every
CRESCENDDI number you compute will be an artefact.

In [3]:
_ = check_crescenddi_sets(g)

  cres_train.npy         n=2834   overlap with OGB edges=0
  cres_val.npy           n=404    overlap with OGB edges=0
  cres_test.npy          n=811    overlap with OGB edges=0
  cres_dd_train.npy      n=1853   overlap with OGB edges=0
  cres_dd_val.npy        n=39     overlap with OGB edges=0
  cres_dd_test.npy       n=172    overlap with OGB edges=0

  pair-level union: 4049   drug-disjoint union: 2064
  shared pairs: 2064 (100% of dd)
  High overlap confirms: same pairs, two splitting strategies.


---
## 2. The confound, before we name it

Mean endpoint degree, per set. Look at this before reading any AUROC.

In [4]:
sets = [("positives (test)", "ogb_test_pos.npy"),
        ("random negatives", "random_neg_test.npy"),
        ("degree-matched",   "degmatched_neg_test.npy"),
        ("CRESCENDDI",       "cres_test.npy")]

print(f"{'set':<22}{'n':>10}{'mean deg':>12}{'median deg':>12}")
for name, fn in sets:
    arr = load_pairs(fn)
    d = np.concatenate([g.deg[arr[:, 0]], g.deg[arr[:, 1]]])
    print(f"{name:<22}{len(arr):>10,}{d.mean():>12.1f}{np.median(d):>12.1f}")

set                            n    mean deg  median deg
positives (test)         133,117       793.8       788.0
random negatives         133,117       453.8       368.0
degree-matched               810       962.2       925.0
CRESCENDDI                   811       966.9       925.0


Random negatives sit at a much lower mean degree than the positives. So a model can
separate the two classes on degree alone, without learning anything about drugs.

CRESCENDDI pairs sit *higher* than the positives — which means the degree signal does
not merely stop working there, it **inverts**.

The degree-matched set exists to remove this. Same construction as the random
negatives, but drawn to reproduce the degree profile of the CRESCENDDI test pairs.

---
## 3. The central result

A logistic regression on **two numbers** — the log-degrees of the two drugs. No
embeddings, no graph convolution, no GPU. Evaluated on the same three sets as the GCN.

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(0)
feats = lambda p: np.sort(g.log_deg[p], axis=1)   # sorted => symmetric

pos = load_pairs("ogb_train_pos.npy")
neg = load_pairs("random_neg_train.npy")
pos = pos[rng.choice(len(pos), 50_000, replace=False)]
neg = neg[rng.choice(len(neg), 50_000, replace=False)]

X = np.vstack([feats(pos), feats(neg)])
y = np.r_[np.ones(len(pos)), np.zeros(len(neg))]
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(X, y)

for name, spec in EVAL_SETS.items():
    ep, en = load_split(spec)
    Xe = np.vstack([feats(ep), feats(en)])
    ye = np.r_[np.ones(len(ep)), np.zeros(len(en))]
    print(f"{name:<18}{roc_auc_score(ye, clf.predict_proba(Xe)[:, 1]):.4f}")

ogb               0.8799
degree_matched    0.3279
crescenddi        0.3288


Expected, roughly: **0.880 / 0.328 / 0.329**.

Two numbers per pair reach 0.880 on the benchmark. The full GCN, with three
convolution layers over 768-dimensional PubMedBERT embeddings, reaches 0.901. The
deep model is worth **0.02** over counting neighbours.

And on the controls, degree alone lands *below chance* — the inversion the degree
table predicted.

A capacity control settles the obvious objection. Adding all 768 embedding dimensions
on top of the two degree features moves the benchmark from 0.880 to 0.890. The
features are not doing the work the architecture implies.

---
## 4. The trained model

Load the checkpoint and score the same three sets. The feature file must match the
checkpoint name: `gcn_random_pubmedbert.pt` was trained on **raw** PubMedBERT, so
passing `pmbwhite` here would produce meaningless embeddings with no error raised.

In [6]:
z = embeddings_from_checkpoint(CKPT_DEFAULT, g, "pubmedbert")
print("z:", z.shape)

learned = stored_scores_auroc("random_pubmedbert_seed0")
posthoc = evaluate_decoders(z)

key = {"ogb": "random", "degree_matched": "degree_matched", "crescenddi": "crescenddi"}
print(f"\n{'eval set':<18}{'learned MLP':>13}{'dot':>9}{'cosine':>9}")
for ev, r in posthoc.items():
    lm = learned.get(key[ev])
    print(f"{ev:<18}{(f'{lm:.3f}' if lm else '  -'):>13}"
          f"{r['dot']:>9.3f}{r['cosine']:>9.3f}")

z: (4267, 256)

eval set            learned MLP      dot   cosine
ogb                       0.901    0.876    0.876
degree_matched            0.664    0.400    0.668
crescenddi                0.514    0.392    0.562


The `learned MLP` column is the real model: **0.901 / 0.664 / 0.514**.

The 0.237 gap between the first two is what degree was contributing. CRESCENDDI stays
near chance even for the best decoder, *below* the degree-matched figure — so the
degree shortcut explains part of the benchmark score but not the failure to transfer.
The rest is source mismatch between DrugBank edges and clinical compendia.

The `dot` and `cosine` columns are post-hoc rescorings of the same embeddings, which
isolate where the degree signal travels. Next cell explains the mechanism.

### Why: the norm channel

Under a dot-product decoder the score factorises:

$$\text{score}(i,j) = \lVert z_i \rVert \cdot \lVert z_j \rVert \cdot \cos(\theta_{ij})$$

Two channels. The angle is pairwise and can encode pharmacology. The **norm** is
per-node and can only encode popularity. Graph convolution sums over neighbours, so a
drug with 2,234 neighbours accumulates a larger activation than one with 1 — the norm
picks up degree whether anyone wants it to or not.

The untrained control is the important part of this cell.

In [10]:
z_untrained = embeddings_untrained(g, "pubmedbert")

_ = norm_degree_diagnostic(z, g, "trained")
_ = norm_degree_diagnostic(z_untrained, g, "untrained")


[trained]  Pearson=0.756  Spearman=0.844  R2=0.571
  norm mean=4.33  std=2.59  cv=0.598
  mean norm per degree decile:
  1.48  0.95  1.99  3.56  4.12  4.90  5.74  5.93  6.71  7.88
  top/bottom decile ratio: 5.34

[untrained]  Pearson=0.872  Spearman=0.929  R2=0.761
  norm mean=17.52  std=9.35  cv=0.534
  mean norm per degree decile:
  3.48  5.07  9.45  15.08  17.32  20.52  22.21  23.82  26.74  31.23
  top/bottom decile ratio: 8.98


Trained: r = 0.756. Untrained, random weights: r = **0.872**.

The coupling is there *before training*. It follows from neighbourhood aggregation,
not from optimisation — training reduces it and never removes it.

This matters for how you write it up. The model **inherits** the degree channel; it
does not learn to exploit it. Phrasing it the other way is a claim the data does not
support, and the untrained control is exactly what a reviewer will ask for.

In [11]:
plot_norm_vs_degree([("trained", z), ("untrained", z_untrained)], g,
                    RESULTS / "norm_vs_degree_handover.png")

figure -> /home/renku/work/data/results/norm_vs_degree_handover.png


---
## 5. Reproducing a training run

```bash
python 05_gcn_experiment.py --features pubmedbert --train-neg random --seed 0
python 05_gcn_experiment.py --sweep --seeds 5     # the full campaign
```

To check the pipeline works without a GPU, subsample the training pairs. The absolute
AUROCs come out lower, but the **ordering** of the three sets and the size of the
spread are preserved — and that is what the experiment is about. `--no-save` keeps it
from overwriting the archived scores.

```bash
python 05_gcn_experiment.py --features pubmedbert --train-neg random \
    --max-train-pairs 50000 --epochs 150 --no-save
```

Two design choices in that script that look like mistakes and are not:

- **The last epoch is reported, not a validation-selected one.** Validation AUROC
  saturates around epoch 15 while control-set AUROC keeps improving until ~300. Early
  stopping on validation stops exactly when the interesting part starts.
- **Imbalance is handled by weighting the loss, never by resampling.** Upsampling the
  verified negatives was tried and degraded validation.